In [ ]:
# load modules
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from scipy.special import gammaln
from scipy.stats import pearsonr, spearmanr
from glob import glob
import os

In [ ]:
# set consts
proj_dir = '/mnt/labworlds/Hayden/Hayden_Lab/speech_247'
data_dir = f'{proj_dir}/anilu_comparison'
patient_list = glob(f'{data_dir}/Y*')
patient_list = [os.path.basename(item) for item in patient_list]

In [ ]:
# get data by patient
gpt_embeddings = {}
for patient in patient_list:
    try:
        print("starting patient:", patient)
        embedding_path = f'{data_dir}/{patient}/embeddings/word_embeddings_gpt2.npy'
        gpt_embedding = np.load(embedding_path, mmap_mode='r')
        layer_36 = np.array(gpt_embedding[:, -1])
        gpt_embeddings[patient] = layer_36
    except Exception as e:
        print("couldn't load for patient:", e)

In [ ]:
# get spike counts by patient
spike_counts = {}

for patient in gpt_embeddings.keys():
    counts_path = f'{data_dir}/{patient}/spikes/word_spikes.npy'
    counts = np.load(counts_path)
    spike_counts[patient] = counts

In [ ]:
# get spike durs by patient
spike_durs = {}
for patient in gpt_embeddings.keys():
    durs_path = f'{data_dir}/{patient}/words/word_info.csv'
    word_df = pd.read_csv(durs_path, index_col=0)
    durs = word_df['Duration'].values
    eps = 1e-6
    log_durs = np.log(np.maximum(durs, eps))
    spike_durs[patient] = log_durs

In [ ]:
def nan_clean_XY(X, Y, durations=None):
    """
    1. Drop rows where *all* X values are NaN or *all* Y values are NaN.
    2. Keep everything else (including Y with some NaNs).
    """
    mask_keep = ~np.isnan(X).all(axis=1)
    mask_keep_y = ~np.isnan(Y).all(axis=1)
    if durations is not None:
        return X[mask_keep & mask_keep_y], Y[mask_keep & mask_keep_y], durations[mask_keep & mask_keep_y]
    return X[mask_keep & mask_keep_y], Y[mask_keep & mask_keep_y]

In [ ]:
def impute_Y_all(Y):
    """
    Impute NaNs in Y using per-channel nanmean from training set only.
    Returns imputed copies of Y_train and Y_test.
    """
    Y_imp = Y.copy()
    # Compute per-channel mean ignoring NaNs
    channel_means = np.nanmean(Y_imp, axis=0)
    # Replace NaN means with 0 if an entire channel is NaN in train
    channel_means = np.where(np.isnan(channel_means), 0.0, channel_means)
    # get channel_means as int
    channel_means = np.round(channel_means).astype(int)
    # Fill NaNs
    inds = np.where(np.isnan(Y_imp))
    Y_imp[inds] = np.take(channel_means, inds[1])
    return Y_imp

In [ ]:
# =============== Utilities ===============


def to_device(x, device):
    return torch.as_tensor(x, dtype=torch.float32, device=device)

def standardize_fit(X):
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0, ddof=0)
    sd[sd == 0] = 1.0
    return mu, sd

def standardize_apply(X, mu, sd):
    return (X - mu) / sd

def backtransform_coef(w_std_np, b_std, mu_np, sd_np):
    # w_raw = w_std / sd ;  intercept_raw = b_std - sum(mu * w_raw)
    w_raw = w_std_np / sd_np[:, np.newaxis]
    intercept_raw = b_std - np.dot(mu_np, w_raw)
    return w_raw, intercept_raw

def poisson_ll_per_neuron(y_true, mu_pred):
    # y_true, mu_pred: (N, K)
    mu = np.clip(mu_pred, 1e-10, None)
    # sum over time, keep neurons
    return (y_true * np.log(mu) - mu - gammaln(y_true + 1)).sum(axis=0)  # (K,)

def pseudo_r2(ll_model, ll_null):
    return 1.0 - (ll_model / ll_null)

In [ ]:
# Hutchinson approximation of trace( B A^{-1} ), where:
#   B = X^T W X, A = B + alpha I, W = diag(mu_train)
def approx_edf_ridge_poisson_torch(Xs_t, mu_t, alpha, n_probe=64):
    # Xs_t: (n,d) standardized train tensor; mu_t: (n,)
    d = Xs_t.shape[1]
    XT_W = (Xs_t.T * mu_t)              # d x n
    B = XT_W @ Xs_t                     # d x d
    A = B + alpha * torch.eye(d, device=Xs_t.device)
    z = torch.randn(d, n_probe, device=Xs_t.device)
    v = torch.linalg.solve(A, z)        # A^{-1} z
    Bv = B @ v
    # trace(B A^{-1}) ≈ mean_i z_i^T (B A^{-1}) z_i
    tr_est = torch.sum(z * Bv, dim=0).mean()
    return float(tr_est.item())

In [ ]:
def approx_edf_ridge_poisson_torch_batched_vectorized(
    Xs_t,        # (n, d)
    mu_t,        # (n, K)
    alpha,       # float OR (K,)
    n_probe=64,
):
    n, d = Xs_t.shape
    K = mu_t.shape[1]
    device = Xs_t.device

    if not torch.is_tensor(alpha) and isinstance(alpha, float):
        alpha = torch.full((K,), float(alpha), device=device)
    elif not torch.is_tensor(alpha) and isinstance(alpha, np.ndarray):
        alpha = torch.tensor(alpha, dtype=torch.float32, device=device)
    else:
        alpha = alpha.to(device)

    # Build B_k = X^T diag(mu_k) X  → (K, d, d)
    # First: X^T * mu  → (K, d, n)
    XT_W = Xs_t.T[None, :, :] * mu_t.T[:, None, :]   # (K, d, n)
    B = XT_W @ Xs_t                                  # (K, d, d)

    I = torch.eye(d, device=device)
    A = B + alpha[:, None, None] * I                 # (K, d, d)

    z = torch.randn(K, d, n_probe, device=device)
    v = torch.linalg.solve(A, z)                     # (K, d, n_probe)
    Bv = B @ v                                       # (K, d, n_probe)

    # Hutchinson trace estimate per neuron
    tr_est = (z * Bv).sum(dim=1).mean(dim=1)         # (K,)

    return tr_est


In [ ]:
import torch.nn as nn
import torch

# =============== Poisson Ridge model ===============

class PoissonRidgeTorchBatched(nn.Module):
    def __init__(self, d, K, alpha=1.0):
        super().__init__()
        self.W = nn.Parameter(torch.zeros(d, K))
        self.b = nn.Parameter(torch.zeros(K))
        
        # alpha can be float or array-like length K
        alpha_t = torch.as_tensor(alpha, dtype=torch.float32)
        if alpha_t.ndim == 0:
            alpha_t = alpha_t.expand(K)        # (K,)
        else:
            assert alpha_t.shape == (K,), f"alpha must be scalar or shape (K,), got {tuple(alpha_t.shape)}"

        # buffer => moves with .to(device), saved in state_dict, not trainable
        self.register_buffer("alpha", alpha_t)

    def set_params(self, w0=None, b0=None):
        with torch.no_grad():
            if w0 is not None:
                self.W.copy_(w0)
            if b0 is not None:
                self.b.copy_(b0)

    def forward(self, X, offset=None):
        eta = X @ self.W + self.b
        if offset is not None:
            eta = eta + offset[:, None]
        return torch.exp(eta)  # μ = exp(η) (log link)

    def loss(self, X, y, offset=None):
        mu = self.forward(X, offset)
        nll = torch.sum(mu - y * torch.log(mu.clamp_min(1e-10)))
        reg = 0.5 * torch.sum(self.alpha * (self.W**2).sum(dim=0))
        return nll + reg

In [ ]:
from torch.optim import LBFGS, Adam
# =============== Fitter (LBFGS or Adam+LBFGS) ===============

def fit_poisson_ridge_lbfgs(
    X_t, y_t, alpha, *,
    offset_t=None,
    init_w=None, init_b=None,
    max_iter=200, tol=1e-6,
    use_full_batch=True, batch_size=65536, adam_steps=None
):
    n, d = X_t.shape
    n, K = y_t.shape
    model = PoissonRidgeTorchBatched(d, K, alpha=alpha).to(X_t.device)
    if init_w is not None or init_b is not None:
        model.set_params(init_w, init_b)

    if use_full_batch:
        optimizer = LBFGS(model.parameters(), lr=1.0, max_iter=max_iter,
                          tolerance_grad=tol, tolerance_change=tol,
                          history_size=10, line_search_fn='strong_wolfe')
        def closure():
            optimizer.zero_grad(set_to_none=True)
            loss = model.loss(X_t, y_t, offset_t)
            loss.backward()
            return loss
        optimizer.step(closure)
    else:
        # Mini-batch Adam warm-up (for memory-constrained cases), then full-batch LBFGS polish
        opt = Adam(model.parameters(), lr=1e-2)
        if adam_steps is None:
            adam_steps = min(2000, max(400, 4 * (n // batch_size + 1)))
        for _ in range(adam_steps):
            idx = torch.randint(0, n, (min(batch_size, n),), device=X_t.device)
            loss = model.loss(X_t[idx], y_t[idx], None if offset_t is None else offset_t[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        optimizer = LBFGS(model.parameters(), lr=1.0, max_iter=max_iter//2,
                          tolerance_grad=tol, tolerance_change=tol,
                          history_size=10, line_search_fn='strong_wolfe')
        def closure2():
            optimizer.zero_grad(set_to_none=True)
            loss = model.loss(X_t, y_t, offset_t)
            loss.backward()
            return loss
        optimizer.step(closure2)

    with torch.no_grad():
        w = model.W.detach().clone()
        b = model.b.detach().clone()
    return model, w, b


In [ ]:
from sklearn.decomposition import PCA
def _prep_X_with_pca(
    X_tr_raw, X_te_raw,
    n_components=100,
):
    """
    Fit standardization + PCA + standardization-in-PCA-space on X_tr_raw only,
    then transform X_te_raw. Returns (Xtr_pca_std, Xte_pca_std, bundle)
    bundle contains everything needed for coef backtransform if desired.
    """
    # standardize in raw space
    mu_raw, sd_raw = standardize_fit(X_tr_raw)
    Xtr_std = standardize_apply(X_tr_raw, mu_raw, sd_raw)
    Xte_std = standardize_apply(X_te_raw, mu_raw, sd_raw)

    # PCA fit on training only
    pca = PCA(n_components=n_components)
    Xtr_pca = pca.fit_transform(Xtr_std)
    Xte_pca = pca.transform(Xte_std)

    # standardize in PCA space (fit on train only)
    mu_pca, sd_pca = standardize_fit(Xtr_pca)
    Xtr_pca_std = standardize_apply(Xtr_pca, mu_pca, sd_pca)
    Xte_pca_std = standardize_apply(Xte_pca, mu_pca, sd_pca)

    bundle = {
        "mu_raw": mu_raw, "sd_raw": sd_raw,
        "pca": pca,
        "mu_pca": mu_pca, "sd_pca": sd_pca,
    }
    return Xtr_pca_std, Xte_pca_std, bundle

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import KFold

def _tune_alpha_inner_cv_gpu_per_neuron(
    X_tr_raw, Y_tr_np, off_tr_np,
    *,
    alphas,
    inner_splits=5,
    seed=42,
    device="cuda",
    n_pca_components=100,
    use_full_batch=True,
    batch_size=65536,
    max_iter=200,
    tol=1e-6,
    # if you need to tune in neuron-chunks, pass Y_tr_np already chunked
):
    """
    Per-neuron alpha tuning (no combinatorial search):
      best_alpha[k] = argmax_a mean_f LL[a, f, k]

    Returns:
      best_alpha_vec: (K,) float numpy
      alpha_ll_mean_vec: (K,) mean LL at chosen alpha
      alpha_ll_std_vec: (K,) std  LL across folds at chosen alpha
      any_wb: warmstart (W, b) from an arbitrary fold/alpha run (optional)
    """
    kf = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)
    splits = list(kf.split(X_tr_raw))

    # Precompute fold tensors; PCA/standardization per fold (correct)
    fold_data = []
    for (tr_f, va_f) in splits:
        X_tr_f, X_va_f = X_tr_raw[tr_f], X_tr_raw[va_f]
        Y_tr_f, Y_va_f = Y_tr_np[tr_f], Y_tr_np[va_f]  # (n_f, K)
        off_tr_f = off_va_f = None
        if off_tr_np is not None:
            off_tr_f, off_va_f = off_tr_np[tr_f], off_tr_np[va_f]

        Xtr_pca_std, Xva_pca_std, _ = _prep_X_with_pca(
            X_tr_f, X_va_f, n_components=n_pca_components
        )

        fold_data.append({
            "Xtr_t": to_device(Xtr_pca_std, device),          # (n_tr, d)
            "Ytr_t": to_device(Y_tr_f, device),               # (n_tr, K)
            "Xva_t": to_device(Xva_pca_std, device),          # (n_va, d)
            "Yva_np": Y_va_f.astype(np.float64, copy=False),  # (n_va, K) for LL
            "off_tr_t": to_device(off_tr_f, device) if off_tr_f is not None else None,
            "off_va_t": to_device(off_va_f, device) if off_va_f is not None else None,
        })

    alphas_desc = np.array(sorted(alphas, reverse=True), dtype=float)
    K = Y_tr_np.shape[1]

    # warm-start cache per fold: store (W,b) with shapes (d,K), (K,)
    fold_cache = {i: (None, None) for i in range(len(fold_data))}

    # scores[a_idx, f_idx, k]
    scores = np.empty((len(alphas_desc), len(fold_data), K), dtype=np.float64)

    for a_idx, a in enumerate(alphas_desc):
        for fidx, fd in enumerate(fold_data):
            W0, b0 = fold_cache[fidx]

            # you need a batched fitter that fits all K neurons simultaneously
            model, W, b = fit_poisson_ridge_lbfgs(
                fd["Xtr_t"], fd["Ytr_t"],
                alpha=float(a),                       # shared within this run
                offset_t=fd["off_tr_t"],
                init_w=W0, init_b=b0,
                max_iter=max_iter, tol=tol,
                use_full_batch=use_full_batch, batch_size=batch_size,
            )
            fold_cache[fidx] = (W, b)

            with torch.no_grad():
                mu_va = model(fd["Xva_t"], fd["off_va_t"]).clamp_min(1e-10).cpu().numpy()  # (n_va, K)

            scores[a_idx, fidx, :] = poisson_ll_per_neuron(fd["Yva_np"], mu_va)

    # mean LL per alpha per neuron: (A,K)
    mean_ll = scores.mean(axis=1)
    std_ll  = scores.std(axis=1, ddof=0)

    best_a_idx = np.argmax(mean_ll, axis=0)                  # (K,)
    best_alpha_vec = alphas_desc[best_a_idx].astype(float)   # (K,)
    alpha_ll_mean_vec = mean_ll[best_a_idx, np.arange(K)]
    alpha_ll_std_vec  = std_ll[best_a_idx, np.arange(K)]

    any_wb = next(iter(fold_cache.values()))
    return best_alpha_vec, alpha_ll_mean_vec, alpha_ll_std_vec, any_wb

In [ ]:
def run_poisson_ridge_gpu_nested_cv(
    X_full, y, *,
    seed=42,
    outer_splits=5,
    inner_splits=5,
    alphas=np.logspace(-3, 3, 30),
    n_shuffles=0,                 # set >0 if you want permutation baselines per fold (expensive)
    device="cuda",
    use_full_batch=True,
    batch_size=65536,
    offset_full=None,
    n_probe_edf=64,
    n_pca_components=100,
    all_data_run=False
):
    """
    Nested CV:
      outer K-fold -> unbiased test metrics
      inner K-fold -> tune alpha inside each outer-train

    Returns:
      model_full (fit on ALL data with alpha chosen by inner-CV on ALL data; optional convenience)
      results dict with:
        fold_metrics: list of per-fold dicts
        summary: aggregated mean/std across folds for key metrics
    """
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)

    if np.std(y) == 0 or np.all(y == 0) or not np.all(np.isfinite(y)):
        return None, {"fold_metrics": [], "summary": None}

    kf_outer = KFold(n_splits=outer_splits, shuffle=True, random_state=seed)

    fold_metrics = []
    for fold_i, (tr_idx, te_idx) in enumerate(kf_outer.split(X_full, Y)):
        print("starting fold:", fold_i)
        X_tr_raw, X_te_raw = X_full[tr_idx], X_full[te_idx]
        y_tr_np, y_te_np   = y[tr_idx], y[te_idx]
        off_tr_np = off_te_np = None
        if offset_full is not None:
            off_tr_np, off_te_np = offset_full[tr_idx], offset_full[te_idx]

        # ---- inner CV tune on outer-train only ----
        print("picking best alpha (warm starts)")
        best_alpha, alpha_ll_mean, alpha_ll_std, (init_w, init_b) = _tune_alpha_inner_cv_gpu_per_neuron(
            X_tr_raw, y_tr_np, off_tr_np,
            alphas=alphas,
            inner_splits=inner_splits,
            seed=seed + 1000 * fold_i,   # vary deterministically by fold
            device=device,
            n_pca_components=n_pca_components,
            use_full_batch=use_full_batch,
            batch_size=batch_size,
        )

        # ---- fit on full outer-train with best alpha ----
        print("fitting final model")
        Xtr_pca_std, Xte_pca_std, bundle = _prep_X_with_pca(
            X_tr_raw, X_te_raw, n_components=n_pca_components
        )
        X_tr_t = to_device(Xtr_pca_std, device)
        y_tr_t = to_device(y_tr_np, device)
        X_te_t = to_device(Xte_pca_std, device)
        y_te_t = to_device(y_te_np, device)
        off_tr_t = to_device(off_tr_np, device) if off_tr_np is not None else None
        off_te_t = to_device(off_te_np, device) if off_te_np is not None else None

        model, w_std_t, b_std_t = fit_poisson_ridge_lbfgs(
            X_tr_t, y_tr_t, alpha=best_alpha,
            offset_t=off_tr_t,
            init_w=init_w, init_b=init_b,
            max_iter=300, tol=1e-6,
            use_full_batch=use_full_batch, batch_size=batch_size
        )

        with torch.no_grad():
            mu_te = model(X_te_t, off_te_t).clamp_min(1e-10).cpu().numpy()

        ll_real = poisson_ll_per_neuron(y_te_np, mu_te)

        # null model (uses train mean rate)
        if off_tr_np is not None:
            avg_rate = y_tr_np.sum(axis=0) / np.exp(off_tr_np).sum()
            mu_null = avg_rate * np.exp(off_te_np)[:, np.newaxis]
        else:
            avg_rate = y_tr_np.mean(axis=0)
            mu_null = np.full_like(y_te_np, avg_rate, dtype=np.float32)
        ll_null = poisson_ll_per_neuron(y_te_np, mu_null)

        pr2 = pseudo_r2(ll_real, ll_null)
        pear = pearsonr(y_te_np, mu_te)[0] if np.std(mu_te) > 0 else np.full_like(ll_real, np.nan)
        spear = spearmanr(y_te_np, mu_te)[0] if np.std(mu_te) > 0 else np.full_like(ll_real, np.nan)

        # EDF/AIC/BIC: compute EDF on TRAIN; AIC/BIC usually use ll on TEST here (your prior behavior)
        with torch.no_grad():
            mu_tr_fit = model(X_tr_t, off_tr_t)
        edf = approx_edf_ridge_poisson_torch_batched_vectorized(X_tr_t, mu_tr_fit, best_alpha, n_probe=n_probe_edf).cpu().numpy()
        aic = 2 * edf - 2 * ll_real
        bic = np.log(y_te_np.shape[0]) * edf - 2 * ll_real

        # optional coef backtransform (your backtransform_coef currently assumes only PCA-space standardization;
        # if you also want to go all the way back through PCA + raw std you’ll need a different function.
        w_std_np = w_std_t.detach().cpu().numpy()
        b_std = b_std_t.detach().cpu().numpy()
        # This matches your existing behavior (backtransform from PCA-standardized -> PCA coords)
        w_pca, intercept_pca = backtransform_coef(w_std_np, b_std, bundle["mu_pca"], bundle["sd_pca"])

        # ---- optional permutation baseline per fold (expensive) ----
        ll_shufs = None
        p_val_ll_xshuf = None
        ll_xshuf_mean = None
        ll_diff = None
        if n_shuffles and n_shuffles > 0:
            print("doing permutation tests")
            ll_shufs = []
            for _ in range(n_shuffles):
                perm = rng.permutation(len(X_full))
                X_shuf_tr_raw = X_full[perm][tr_idx]
                X_shuf_te_raw = X_full[perm][te_idx]

                # IMPORTANT: refit preprocess on shuffled TRAIN only (no leakage)
                Xs_tr, Xs_te, _ = _prep_X_with_pca(
                    X_shuf_tr_raw, X_shuf_te_raw, n_components=n_pca_components
                )
                Xs_tr_t = to_device(Xs_tr, device)
                Xs_te_t = to_device(Xs_te, device)

                m_shuf, _, _ = fit_poisson_ridge_lbfgs(
                    Xs_tr_t, y_tr_t, alpha=best_alpha,
                    offset_t=off_tr_t,
                    max_iter=200, tol=1e-6,
                    use_full_batch=use_full_batch, batch_size=batch_size
                )
                with torch.no_grad():
                    mu_s_te = m_shuf(Xs_te_t, off_te_t).clamp_min(1e-10).cpu().numpy()
                ll_shufs.append(poisson_ll_per_neuron(y_te_np, mu_s_te))
            
            ll_shufs = np.array(ll_shufs)
            ll_xshuf_mean = np.mean(ll_shufs, axis=0)
            ll_diff = ll_real - ll_xshuf_mean
            p_val_ll_xshuf = (np.sum(ll_shufs >= ll_real, axis=0) + 1) / (ll_shufs.shape[0] + 1)

        fold_metrics.append({
            "state_dict": model.state_dict(), 
            "fold": fold_i,
            "best_alpha": best_alpha,
            "alpha_ll_mean": alpha_ll_mean,
            "alpha_ll_std": alpha_ll_std,
            "ll_real": ll_real,
            "ll_null": ll_null,
            "pseudo_r2": pr2,
            "pearson_corr": pear,
            "spearman_corr": spear,
            "edf": edf,
            "aic": aic,
            "bic": bic,
            "ll_shufs": np.array(ll_shufs, dtype=float),
            "ll_xshuf_mean": ll_xshuf_mean,
            "ll_diff": ll_diff,
            "p_val_ll_xshuf": p_val_ll_xshuf,
            "coef_pca_space": w_pca.astype(float),
            # you can store intercept_pca too if useful
        })

    # ---- aggregate summary ----
    def _agg(key):
        vals = np.array([fm[key] for fm in fold_metrics], dtype=float)
        return {"mean": np.nanmean(vals, axis=0), "std": np.nanstd(vals, axis=0)}

    p_val_ll_xshuf = None
    ll_xshuf_mean = None
    if n_shuffles and n_shuffles > 0:
        # observed aggregated statistic
        d_obs = np.array([fm["ll_real"] for fm in fold_metrics], dtype=float)
        T_obs = np.mean(d_obs, axis=0)
    
        # build permutation aggregated statistics, aligned by permutation index
        K = len(fold_metrics)
        B = n_shuffles
    
        # shape: (K, B)
        ll_shufs_mat = np.array([fm["ll_shufs"] for fm in fold_metrics])
    
        T_perm = np.mean(ll_shufs_mat, axis=0)   # shape (N, B,)
    
        # one-sided p-value: how often perm >= obs
        p_val_ll_xshuf = (np.sum(T_perm >= T_obs, axis=0) + 1) / (B + 1)
        ll_xshuf_mean = np.mean(np.concatenate([fm["ll_shufs"] for fm in fold_metrics]), axis=0)

    summary = {
        "outer_splits": int(outer_splits),
        "inner_splits": int(inner_splits),
        "best_alpha": _agg("best_alpha"),
        "ll_real": _agg("ll_real"),
        "pseudo_r2": _agg("pseudo_r2"),
        "pearson_corr": _agg("pearson_corr"),
        "spearman_corr": _agg("spearman_corr"),
        "p_val_ll_xshuf": p_val_ll_xshuf,
        "ll_xshuf_mean": ll_xshuf_mean, 
        "edf": _agg("edf"),
        "aic": _agg("aic"),
        "bic": _agg("bic"),
    }

    # ---- optional convenience fit on ALL data using inner-CV on ALL data ----
    if all_data_run:
        X_all_pca_std, _, bundle_all = _prep_X_with_pca(X_full, X_full, n_components=n_pca_components)
        X_all_t = to_device(X_all_pca_std, device)
        y_all_t = to_device(y, device)
        off_all_t = to_device(offset_full, device) if offset_full is not None else None
    
        model_full, w_full, b_full = fit_poisson_ridge_lbfgs(
            X_all_t, y_all_t, alpha=summary["best_alpha"]["mean"],
            offset_t=off_all_t,
            init_w=init_w, init_b=init_b,
            max_iter=400, tol=1e-6,
            use_full_batch=use_full_batch, batch_size=batch_size
        )
            
        return model_full, {
            "fold_metrics": fold_metrics,
            "summary": summary,
            "best_alpha_full": float(best_alpha_full),
            "bundle_full": bundle_all,  # PCA/std params if you want to transform new data consistently
        }
    else:
        return None, {
            "fold_metrics": fold_metrics,
            "summary": summary,
            "best_alpha_full": None,
            "bundle_full": None,  # PCA/std params if you want to transform new data consistently
        }


In [ ]:
# now get poisson ridge results for each neuron
from joblib import Parallel, delayed
import dill as pickle
import torch
poisson_dfs = {}
poisson_models_all = {}
for patient in gpt_embeddings.keys():
    # skip patients for which we already have results
    res_path = f'{data_dir}/{patient}/results/word_encoding_results_cv.pkl'
    if os.path.exists(res_path):
        print("already have results for", patient)
        continue
    print("starting patient", patient)
    poisson_models = {}
    poisson_res = []
    embeddings = gpt_embeddings[patient]
    durations = spike_durs[patient]
    
    # get results for offset 0
    spikes = spike_counts[patient]
    _, n_neuron = spikes.shape
    X, Y, dur_clean = nan_clean_XY(embeddings, spikes, durations=durations)
    Y = impute_Y_all(Y)
    results = run_poisson_ridge_gpu_nested_cv(X, Y, offset_full=dur_clean, n_shuffles=50)
    # save weights of torch model
    if results is not None:
        df_row = {}
        res_info = results[1]
        df_row["best_alpha_whole"] = np.nan
        df_row["fold"] = np.nan
        df_row["outer_splits"] = res_info["summary"]["outer_splits"]
        df_row["inner_splits"] = res_info["summary"]["inner_splits"]
        df_row["summary"] = False
        for fm in res_info["fold_metrics"]:
            fold_id = fm["fold"]
            poisson_models[f"k_{fold_id}"] = fm["state_dict"]
            for n in range(n_neuron):
                f_row = df_row.copy()
                f_row["neuron_idx"] = n
                f_row["fold_id"] = fold_id
                f_row["best_alpha"] = fm["best_alpha"][n]
                f_row["alpha_ll_mean"] = fm["alpha_ll_mean"][n]
                f_row["alpha_ll_std"] = fm["alpha_ll_std"][n]
                f_row["ll_real"] = fm["ll_real"][n]
                f_row["ll_null"] = fm["ll_null"][n]
                f_row["pseudo_r2"] = fm["pseudo_r2"][n]
                f_row["pearson_corr"] = fm["pearson_corr"][n]
                f_row["spearman_corr"] = fm["spearman_corr"][n]
                f_row["edf"] = fm["edf"][n]
                f_row["aic"] = fm["aic"][n]
                f_row["bic"] = fm["bic"][n]
                f_row["ll_shufs"] = fm["ll_shufs"][:, n]
                f_row["ll_xshuf_mean"] = fm["ll_xshuf_mean"][n]
                f_row["ll_diff"] = fm["ll_diff"][n]
                f_row["p_val_ll_xshuf"] = fm["p_val_ll_xshuf"][n]
                f_row["coef_pca_space"] = fm["coef_pca_space"][:, n]
                poisson_res.append(f_row)
        df_row["summary"] = True
        df_row["ll_shufs"] = np.nan
        for n in range(n_neuron):
            summary_row = df_row.copy()
            summary_row["best_alpha_mean"] = res_info["summary"]["best_alpha"]["mean"][n]
            summary_row["best_alpha_std"] = res_info["summary"]["best_alpha"]["std"][n]
            
            summary_row["ll_real_mean"] = res_info["summary"]["ll_real"]["mean"][n]
            summary_row["ll_real_std"] = res_info["summary"]["ll_real"]["std"][n]
            
            summary_row["pseudo_r2_mean"] = res_info["summary"]["pseudo_r2"]["mean"][n]
            summary_row["pseudo_r2_std"] = res_info["summary"]["pseudo_r2"]["std"][n]
            
            summary_row["pearson_corr_mean"] = res_info["summary"]["pearson_corr"]["mean"][n]
            summary_row["pearson_corr_std"] = res_info["summary"]["pearson_corr"]["std"][n]
            
            summary_row["spearman_corr_mean"] = res_info["summary"]["spearman_corr"]["mean"][n]
            summary_row["spearman_corr_std"] = res_info["summary"]["spearman_corr"]["std"][n]
            
            summary_row["p_val_ll_xshuf"] = res_info["summary"]["p_val_ll_xshuf"][n]
            summary_row["ll_xshuf_mean"] = res_info["summary"]["ll_xshuf_mean"][n]
            
            summary_row["edf_mean"] = res_info["summary"]["edf"]["mean"][n]
            summary_row["edf_std"] = res_info["summary"]["edf"]["std"][n]
            
            summary_row["aic_mean"] = res_info["summary"]["aic"]["mean"][n]
            summary_row["aic_std"] = res_info["summary"]["aic"]["std"][n]
            
            summary_row["bic_mean"] = res_info["summary"]["bic"]["mean"][n]
            summary_row["bic_std"] = res_info["summary"]["bic"]["std"][n]
            
            poisson_res.append(summary_row)
            

    # save to dataframe
    poisson_df = pd.DataFrame(poisson_res)
    # save results
    with open(f'{data_dir}/{patient}/results/word_encoding_results_cv.pkl', 'wb') as f:
        pickle.dump(poisson_df, f)
    torch.save(poisson_models, f'{data_dir}/{patient}/results/word_encoding_results_cv.tar')
    poisson_dfs[patient] = poisson_df
    poisson_models_all[patient] = poisson_models
    

In [ ]:
poisson_df
